In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/policies_contents.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/test_contents.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/claims.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/policies_buildings.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/weather.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/sample_submission.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/customers.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/addresses.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/test_buildings.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/test_motor.parquet
/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/publi

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import r2_score, roc_auc_score, mean_tweedie_deviance

BASE = '/kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public/'
SEED = 42
POWER = 1.5
FLOOR = 0.25                 # prediccion minima = 25% del costo medio por exposicion
SALIDA = 'modelo'            # 'modelo' | 'B'
ESCALA = {'Motor': 1.15, 'Buildings': 1.15, 'Contents': 1.15}  # multiplicador de nivel por cartera

AJUSTE_IBNR = True           # corrige siniestros de 2024 aun no reportados al 31-dic-2024
TENDENCIA = True             # lleva los costos historicos al nivel de 2025
ANIO_OBJETIVO = 2025
VALUACION = pd.Timestamp('2024-12-31')
PMIN = 0.25                  # tope del factor IBNR: maximo 4x sobre un siniestro observado


def rd(name):
    return pd.read_parquet(BASE + name)


customers, addresses, weather, claims = (rd(f'{n}.parquet') for n in
                                         ['customers', 'addresses', 'weather', 'claims'])
claims['loss_date'] = pd.to_datetime(claims['loss_date'])
claims['report_date'] = pd.to_datetime(claims['report_date'])

VAL = {}  # predicciones de validacion 2024 por cartera
CARTERAS = {  # nombre: (archivo train, archivo test, valor de 'line' en claims)
    'Motor':     ('policies_motor',     'test_motor',     'motor'),
    'Buildings': ('policies_buildings', 'test_buildings', 'buildings'),
    'Contents':  ('policies_contents',  'test_contents',  'contents'),
}

WCOLS = [c for c in weather.columns if c not in ('address_id', 'year')]
weather_prom = weather.groupby('address_id')[WCOLS].mean().add_suffix('_prom').reset_index()

# customer_service_contacts: se genera DESPUES del siniestro (fuga); customer_since_year delata el anio
IGNORAR = ['row_id', 'policy_id', 'customer_id', 'address_id', 'period_start', 'period_end',
           'renewal_decision', 'target_costo', 'surname', 'date_of_birth', 'exposure', '_yr',
           'customer_service_contacts', 'customer_since_year']


def dev(y, p):
    return mean_tweedie_deviance(y, np.clip(p, 1e-6, None), power=POWER)


# ---------------------------------------------------------------- IBNR
def curva_reporte(line):
    """Distribucion del retraso (report_date - loss_date) con siniestros de >=18 meses de antiguedad."""
    c = claims[claims['line'] == line]
    ret = (c['report_date'] - c['loss_date']).dt.days
    maduros = ret[(c['loss_date'] <= pd.Timestamp('2023-06-30')) & ret.notna()].clip(lower=0)
    maduros = np.sort(maduros.to_numpy())
    return lambda d: np.searchsorted(maduros, d, side='right') / len(maduros)


# ---------------------------------------------------------------- target por fila
def costo_por_fila(pol, line):
    """Asigna cada siniestro a la fila (policy_id + periodo) cuyo rango contiene su loss_date."""
    p = pol[['policy_id', 'period_start', 'period_end']].copy()
    p['_i'] = np.arange(len(p))
    p['period_start'] = pd.to_datetime(p['period_start'])
    p['period_end'] = pd.to_datetime(p['period_end'])
    c = claims.loc[claims['line'] == line, ['claim_id', 'policy_id', 'loss_date', 'incurred_to_date']].copy()
    bruto = c['incurred_to_date'].sum()
    if AJUSTE_IBNR:
        F = curva_reporte(line)
        prob = np.clip(F((VALUACION - c['loss_date']).dt.days.to_numpy()), PMIN, 1.0)
        c['incurred_to_date'] = c['incurred_to_date'] / prob
        print(f'  [{line}] ajuste IBNR: incurrido total x{c["incurred_to_date"].sum() / bruto:.3f}'
              f' | prob. media de estar reportado en 2024={prob[c["loss_date"].dt.year.to_numpy() == 2024].mean():.3f}')
    m = c.merge(p, on='policy_id')
    m = m[(m['loss_date'] >= m['period_start']) & (m['loss_date'] <= m['period_end'])]
    m = m.drop_duplicates('claim_id')
    s = m.groupby('_i')['incurred_to_date'].sum()
    cobertura = m['incurred_to_date'].sum() / max(c['incurred_to_date'].sum(), 1e-9)
    print(f'  [{line}] siniestros asignados a una fila: {len(m)}/{len(c)} ({cobertura:.1%} del incurrido)')
    return p['_i'].map(s).fillna(0).to_numpy()


# ---------------------------------------------------------------- historial de la poliza
INICIO_DATOS = pd.Timestamp('2021-01-01')


def historial(pol, line):
    """Siniestralidad previa de la MISMA poliza usando solo siniestros ocurridos ANTES del inicio de la fila.
    Se usan ventanas fijas (365 y 730 dias) para que no dependan del anio calendario; si la ventana
    empieza antes de 2021 el dato es desconocido (NaN), no cero."""
    p = pol[['policy_id', 'period_start']].copy()
    p['_i'] = np.arange(len(p))
    p['period_start'] = pd.to_datetime(p['period_start'])
    c = claims.loc[claims['line'] == line, ['policy_id', 'loss_date', 'incurred_to_date']]
    m = p.merge(c, on='policy_id')
    m['dias'] = (m['period_start'] - m['loss_date']).dt.days
    m = m[m['dias'] > 0]
    n = np.arange(len(p))
    h = pd.DataFrame(index=n)
    for d in (365, 730):
        sel = m[m['dias'] <= d].groupby('_i')
        h[f'n_sin_{d}d'] = sel.size().reindex(n).fillna(0)
        h[f'inc_{d}d'] = sel['incurred_to_date'].sum().reindex(n).fillna(0)
        completa = ((p['period_start'] - pd.Timedelta(days=d)) >= INICIO_DATOS).to_numpy()
        h.loc[~completa, [f'n_sin_{d}d', f'inc_{d}d']] = np.nan
    ult = m.groupby('_i')['dias'].min().reindex(n)
    h['dias_ultimo_sin'] = ult.where(ult <= 730)
    return h


# ---------------------------------------------------------------- features
def integrar(df):
    df = df.merge(customers, on='customer_id', how='left').merge(addresses, on='address_id', how='left')
    ini = pd.to_datetime(df['period_start'])
    df['_yr'] = ini.dt.year
    df = df.merge(weather.rename(columns={'year': '_yr'}), on=['address_id', '_yr'], how='left')
    df = df.merge(weather_prom, on='address_id', how='left')
    for c in WCOLS:  # si falta el clima del anio (p. ej. 2025), usar el promedio del domicilio
        df[c] = df[c].fillna(df[c + '_prom'])
    df = df.drop(columns=[c + '_prom' for c in WCOLS])
    df['mes_inicio'] = ini.dt.month
    if 'date_of_birth' in df:
        df['edad_cliente'] = (ini - pd.to_datetime(df['date_of_birth'])).dt.days / 365.25
    if 'year_built' in df:
        df['antiguedad_prop'] = df['_yr'] - df['year_built']
    return df


def a_str(s):
    return s.astype(str).where(s.notna(), np.nan)


def preparar(train_full, test_full):
    cols = [c for c in train_full.columns
            if c not in IGNORAR and not pd.api.types.is_datetime64_any_dtype(train_full[c])]
    X_tr, X_te = train_full[cols].copy(), test_full[cols].copy()
    for c in X_tr.select_dtypes(include=['object', 'category', 'string']).columns:
        cats = pd.Index(a_str(X_tr[c]).dropna().unique()).sort_values()
        X_tr[c] = pd.Categorical(a_str(X_tr[c]), categories=cats)
        X_te[c] = pd.Categorical(a_str(X_te[c]), categories=cats)
    return X_tr, X_te


def diagnostico_deriva(X_tr, X_te):
    """Solo informa: AUC de distinguir train de test y variables que mas distinguen."""
    Xa = pd.concat([X_tr, X_te], ignore_index=True)
    ya = np.r_[np.zeros(len(X_tr)), np.ones(len(X_te))]
    idx = np.random.RandomState(SEED).permutation(len(Xa))
    cut = int(len(Xa) * 0.7)
    adv = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, verbose=-1, n_jobs=-1)
    adv.fit(Xa.iloc[idx[:cut]], ya[idx[:cut]])
    auc = roc_auc_score(ya[idx[cut:]], adv.predict_proba(Xa.iloc[idx[cut:]])[:, 1])
    imp = pd.Series(adv.booster_.feature_importance('gain'), index=Xa.columns)
    print(f'  deriva train-vs-test: AUC={auc:.3f} | variables que mas distinguen: {imp.nlargest(3).index.tolist()}')


# ---------------------------------------------------------------- modelo
PARAMS = dict(objective='tweedie', tweedie_variance_power=POWER, learning_rate=0.03,
              num_leaves=15, min_child_samples=200, colsample_bytree=0.8, subsample=0.8,
              subsample_freq=1, reg_lambda=10.0, cat_smooth=50, cat_l2=10, min_data_per_group=200,
              random_state=SEED, n_jobs=-1, verbose=-1)


def offset(e, tasa):  # log(exposicion) + log(tasa media): el arbol solo aprende el ajuste relativo
    return np.log(np.clip(e, 1e-6, None)) + np.log(tasa)


def modelar(nombre, f_train, f_test, line):
    print(f'Entrenando {nombre}...')
    df_train, df_test = rd(f_train + '.parquet'), rd(f_test + '.parquet')
    df_train = df_train.reset_index(drop=True)
    df_train['target_costo'] = costo_por_fila(df_train, line)

    train_full, test_full = integrar(df_train), integrar(df_test)
    assert len(test_full) == len(df_test), 'un merge duplico filas del test'
    for col_h, v in historial(df_train, line).items():
        train_full[col_h] = v.to_numpy()
    for col_h, v in historial(df_test, line).items():
        test_full[col_h] = v.to_numpy()
    X_tr, X_te = preparar(train_full, test_full)
    diagnostico_deriva(X_tr, X_te)

    expo = train_full['exposure'].to_numpy()
    expo_te = test_full['exposure'].to_numpy()
    prima = train_full['written_premium'].fillna(0).to_numpy()
    prima_te = test_full['written_premium'].fillna(0).to_numpy()
    ok = expo > 0
    anio = train_full['_yr'].to_numpy()
    y = train_full['target_costo'].to_numpy().astype(float)

    tasa_anio = pd.Series(y[ok]).groupby(anio[ok]).sum() / pd.Series(expo[ok]).groupby(anio[ok]).sum()
    print('  costo por anio-poliza historico:', {int(a): round(float(v), 1) for a, v in tasa_anio.items()})
    if TENDENCIA:
        g = np.exp(np.polyfit(tasa_anio.index.to_numpy(float), np.log(tasa_anio.to_numpy()), 1)[0]) - 1
        g = float(np.clip(g, -0.05, 0.15))
        y = y * (1 + g) ** (ANIO_OBJETIVO - anio)
        print(f'  tendencia anual estimada g={g:.1%} -> costos llevados a nivel {ANIO_OBJETIVO}')
    pos = y[ok][y[ok] > 0]

    # Validacion temporal: entrenar hasta 2023, validar en 2024 (imita la prediccion de 2025)
    va = (anio == anio.max()) & ok
    tr = (~va) & ok
    y_va = y[va]
    k = y[tr].sum() / max(prima[tr].sum(), 1e-9)
    ref_va = k * prima[va]                                   # B: prima escalada al nivel de costo
    C_va = (y[tr].sum() / expo[tr].sum()) * expo[va]         # C: costo medio constante x exposicion
    base = dev(y_va, np.full_like(y_va, y_va.mean()))

    # Se entrena con costo recortado (q) y se repone la masa; se elige por devianza
    resultados = []
    for q in (0.99, 0.999, 1.0):
        cap = np.quantile(pos, q) if q < 1 else pos.max()
        yc = np.minimum(y, cap)
        carga = y[tr].sum() / max(yc[tr].sum(), 1e-9)
        tasa = yc[tr].sum() / expo[tr].sum()
        m = lgb.LGBMRegressor(n_estimators=3000, **PARAMS)
        m.fit(X_tr[tr], yc[tr], init_score=offset(expo[tr], tasa),
              eval_set=[(X_tr[va], yc[va])], eval_init_score=[offset(expo[va], tasa)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pred_va = np.clip(np.exp(m.predict(X_tr[va], raw_score=True) + offset(expo[va], tasa)), 0, cap) * carga
        mejor = (-np.inf, 1.0)
        for a in np.linspace(0, 1, 11):
            bl = np.maximum(a * pred_va + (1 - a) * ref_va, FLOOR * C_va)
            sk = 1 - dev(y_va, bl) / base
            if sk > mejor[0]:
                mejor = (sk, a)
        sk, a = mejor
        print(f'  cap q={q} ({cap:.0f}) | iter={m.best_iteration_} | skill devianza: modelo={1 - dev(y_va, pred_va) / base:.4f}'
              f' prima={1 - dev(y_va, ref_va) / base:.4f} mezcla(a={a:.1f})={sk:.4f}'
              f' | pred/real={pred_va.sum() / max(y_va.sum(), 1e-9):.3f}')
        resultados.append((sk, q, m.best_iteration_, a, pred_va))

    sk, q, it, a_ch, pred_ch = max(resultados, key=lambda r: r[0])
    print(f'  >> elegido: q={q}, iteraciones={it}, peso modelo a={a_ch:.1f}, skill devianza validacion={sk:.4f}')
    VAL[nombre] = dict(y=y_va, A=prima[va], B=ref_va, C=C_va, modelo=pred_ch,
                       mezcla=np.maximum(a_ch * pred_ch + (1 - a_ch) * ref_va, FLOOR * C_va))

    # Refit con todo el historico usando la configuracion elegida
    cap = np.quantile(pos, q) if q < 1 else pos.max()
    yc = np.minimum(y, cap)
    carga = y[ok].sum() / max(yc[ok].sum(), 1e-9)
    tasa = yc[ok].sum() / expo[ok].sum()
    modelos = []
    for sd in (42, 7, 2026):  # promedio de 3 semillas: predicciones mas estables
        final = lgb.LGBMRegressor(n_estimators=max(int(it * 1.1), 50), **dict(PARAMS, random_state=sd))
        final.fit(X_tr[ok], yc[ok], init_score=offset(expo[ok], tasa))
        modelos.append(np.exp(final.predict(X_te, raw_score=True) + offset(expo_te, tasa)))
    modelo = np.clip(np.mean(modelos, axis=0), 0, cap) * carga

    B_te = (y[ok].sum() / max(prima[ok].sum(), 1e-9)) * prima_te
    C_te = (y[ok].sum() / expo[ok].sum()) * expo_te
    opciones = {'modelo': np.maximum(a_ch * modelo + (1 - a_ch) * B_te, FLOOR * C_te), 'B': B_te}
    pred = np.clip(opciones[SALIDA] * ESCALA[nombre], 0, None)
    print(f'  nivel 2025: costo por anio-poliza modelo={modelo.sum() / expo_te.sum():.1f}'
          f' | enviado ({SALIDA} x{ESCALA[nombre]})={pred.sum() / expo_te.sum():.1f}')
    return pd.DataFrame({'row_id': test_full['row_id'].to_numpy(), 'prediction': pred})


preds = [modelar(n, *args) for n, args in CARTERAS.items()]

Y = np.concatenate([v['y'] for v in VAL.values()])
base_dev = dev(Y, np.full_like(Y, Y.mean()))
print('\n=== Validacion 2024 agregada (las 3 carteras juntas, como el leaderboard) ===')
for nombre in ('C', 'A', 'B', 'modelo', 'mezcla'):
    P = np.concatenate([v[nombre] for v in VAL.values()]).astype(float)
    print(f'{nombre:7s} skill devianza={1 - dev(Y, P) / base_dev:.4f}')

sub = pd.concat(preds, ignore_index=True)
assert sub['prediction'].ge(0).all() and sub['row_id'].is_unique
sub.to_csv('submission.csv', index=False)
print(f'\n¡Archivo listo! SALIDA={SALIDA} ESCALA={ESCALA}', sub.shape)

Entrenando Motor...
  [motor] ajuste IBNR: incurrido total x1.015 | prob. media de estar reportado en 2024=0.946
  [motor] siniestros asignados a una fila: 112309/112606 (99.7% del incurrido)
  deriva train-vs-test: AUC=0.999 | variables que mas distinguen: ['region', 'n_sin_730d', 'dry_summer_index']
  costo por anio-poliza historico: {2021: 444.0, 2022: 468.4, 2023: 455.3, 2024: 388.7}
  tendencia anual estimada g=-4.2% -> costos llevados a nivel 2025
  cap q=0.99 (35647) | iter=243 | skill devianza: modelo=0.0762 prima=0.0168 mezcla(a=1.0)=0.0762 | pred/real=1.077
  cap q=0.999 (109495) | iter=233 | skill devianza: modelo=0.0710 prima=0.0168 mezcla(a=0.9)=0.0715 | pred/real=1.186
  cap q=1.0 (10952117) | iter=291 | skill devianza: modelo=0.0675 prima=0.0168 mezcla(a=0.9)=0.0693 | pred/real=1.231
  >> elegido: q=0.99, iteraciones=243, peso modelo a=1.0, skill devianza validacion=0.0762
  nivel 2025: costo por anio-poliza modelo=477.2 | enviado (modelo x1.15)=548.8
Entrenando Building